In [ ]:
import torch
from torch import nn
import math

"""
Model: ChatGPT 6.1 Sol Extra-High
Can you provide an extension of RMS Norm for the complex numbers? Give a proper citation to any papers cited. Provide the implementation in pytorch.

\begin{response}
citation: 
Zhang, B., & Sennrich, R. (2019).
Root Mean Square Layer Normalization.
Advances in Neural Information Processing Systems, 32.
https://arxiv.org/abs/1910.07467

Contribution:
The cited paper introduces real-valued RMSNorm. This response
proposes a complex-valued extension using squared complex
magnitudes, with normalization across the channel dimension.
The extension and implementation are generated adaptations.
"""
class ComplexRMSNorm(nn.Module):
    def __init__(self,eps=1e-6):
        super().__init__()
        self.eps=eps

    def forward(self,x):
        power = x.real.square()+x.imag.square()
        return x*torch.rsqrt(power.mean(dim=1,keepdim=True)+self.eps)

# \end{response}

# batch, channels, n, m | cat adds to channels
def cat(*xs): return torch.cat(xs,dim=1)

class EMLConv(nn.Module):
    def __init__(self, in_channels, out_channels, log_channels=None, transpose=False, \
                 dtype=torch.complex64, device=None, gain=.2, eps=1e-6):
        super().__init__()
        conv = nn.ConvTranspose2d if transpose else nn.Conv2d
        k,s,p = (4,2,1) if transpose else (3,1,1)
        kw = dict(kernel_size=k,stride=s,padding=p,bias=True,dtype=dtype,device=device)
        self.exp_filter = conv(in_channels,out_channels,**kw)
        self.log_filter = conv(in_channels if log_channels is None else log_channels,
                             out_channels,**kw)
        self.norm = ComplexRMSNorm(eps)
        with torch.no_grad():
            self.exp_filter.weight.mul_(gain)
            self.log_filter.weight.mul_(gain)
            self.exp_filter.bias.zero_()
            self.log_filter.bias.fill_(math.e)

    def forward(self,x:torch.tensor,log_input:torch.tensor=None):
        dtype = self.exp_filter.weight.dtype
        y = x if log_input is None else log_input
        u = self.exp_filter(x.to(dtype=dtype))
        v = self.log_filter(y.to(dtype=dtype))
        return self.norm(torch.exp(u)-torch.log(v))

class ConvEmbedding(nn.Module):
    def __init__(self, v_channels, out_channels, **kw):
        super().__init__()
        temp = 3*v_channels
        self.first = nn.ModuleDict({
            'NN': EMLConv(temp, out_channels, **kw),
            'NV': EMLConv(temp, out_channels, log_channels=v_channels, **kw),
            'VN': EMLConv(v_channels, out_channels, log_channels=temp, **kw),
            'VV': EMLConv(v_channels, out_channels, **kw), # out_channels = 1? or would Conv2D w/ 3 filters have better RGB variability; keeping 3 + no exception for self.second
        })
        self.second = nn.ModuleDict({
            layer: EMLConv(inp_mod*out_channels, out_channels, **kw)
            for layer, inp_mod in (('V3N', 2), ('V3NNV', 3), ('V3NNVNN', 4))
        })
        self.third = nn.ModuleDict({
            layer: EMLConv(inp_mod*out_channels, out_channels, **kw)
            for layer, inp_mod in (('NV', 2), ('NVNN', 3))
        })
        self.fourth = EMLConv(2*out_channels, out_channels, **kw)

    def forward(self, n, v):
        n2 = self.first['NN'](n)
        nv = self.first['NV'](n, v) # exp(Noise) - ln(embedding_vector)
        vn = self.first['VN'](v, n) # opposite
        v2 = self.first['VV'](v)
        v3n = self.second['V3N'](cat(v2, vn))
        v3n2v = self.second['V3NNV'](cat(v2, vn, nv))
        n3n2vn2 = self.second['V3NNVNN'](cat(v2, vn, nv, n2))
        nv2 = self.third['NV'](cat(n3n2vn2, v3n2v))
        nvn2 = self.third['NVNN'](cat(n3n2vn2, v3n2v, v3n))
        return self.fourth(cat(nv2, nvn2))

class ColorFocus(nn.Module):
    def __init__(self, color_channels, out_channels, **kw):
        super().__init__()
        # initially thought of splitting color_channels into r, g, b but input is still abstract so an additional 3->1 won't hurt + more learned activation functions
        # Abstract for color in range(color_channels)
        # double for loop: 2nd order, triple 3rd -> add n-order before terniquete to all then add stride to selection ?
        self.colors = nn.ModuleDict({color: EMLConv(color_channels, out_channels, **kw) for color in 'RGB'})
        self.pair_colors = nn.ModuleDict({color: EMLConv(2*out_channels, out_channels, **kw) for color in ('RG', 'RB', 'GB')})
        self.second_order = nn.ModuleDict({color: EMLConv(2*out_channels, out_channels, **kw) for color in ('RGGB', 'RBBG', 'BRRG')})
        self.all = EMLConv(3*out_channels, out_channels, **kw)
        self.to_color = nn.ModuleDict({color: EMLConv(4*out_channels, out_channels, **kw) for color in 'rgb'})

    def forward(self, x:torch.tensor):
        r, g, b = (self.colors[c](x) for c in 'RGB')
        rg = self.pair_colors['RG'](cat(r, g))
        rb = self.pair_colors['RB'](cat(r, b))
        gb = self.pair_colors['GB'](cat(g, b))
        # RGGB, RBBG, BRRG | independent of order eml(cat(a,b)) = eml(cat(b, a))
        rggb = self.second_order['RGGB'](cat(rg, gb))
        rbbg = self.second_order['RBBG'](cat(rb, gb))
        brrg = self.second_order['BRRG'](cat(rb, rg))
        all = self.all(cat(rggb, rbbg, brrg))

        # built in skip connection | use without repeating middle; skip 2, force second order eml ?
        R = self.to_color['r'](cat(r, rggb, rbbg, all))
        B = self.to_color['b'](cat(b, rggb, brrg, all))
        G = self.to_color['g'](cat(g, rbbg, brrg, all))

        return cat(R, B, G)

class EMLUpcycle(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, start_size, out_size, device=None, **kw):
        super().__init__()
      
        blocks = [ColorFocus(in_channels, mid_channels//3, device=device, **kw)]
        curr_size = start_size
        while 2*curr_size < out_size: # not the final convolution -> up
            blocks.append(EMLConv(3*(mid_channels//3), 3*(mid_channels//3), transpose=True, device=device, **kw))
            blocks.append(ColorFocus(3*(mid_channels//3), mid_channels//3, device=device, **kw))
            curr_size *= 2
        blocks.append(EMLConv(3*(mid_channels//3), out_channels, transpose=True, device=device, **kw))
        self.layers = nn.Sequential(*blocks)

    def forward(self, x:torch.tensor):
        return self.layers(x)

class CRConvUpcycle(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, start_size, out_size, device=None, k=4,s=2,p=1, **kw):
        super().__init__()
        # real + complex components
        blocks = [nn.Conv2d(2*in_channels, mid_channels, 3,1,1, device=device, **kw),
                  nn.BatchNorm2d(mid_channels, device=device), nn.ReLU(inplace=True)]
        curr_size = start_size
        while 2*curr_size < out_size:
            blocks.append(nn.ConvTranspose2d(mid_channels, mid_channels, k,s,p, device=device,**kw))
            blocks.append(nn.BatchNorm2d(mid_channels, device=device, **kw))
            blocks.append(nn.ReLU(inplace=True))
            curr_size *= 2
        blocks.append(nn.ConvTranspose2d(mid_channels, out_channels, k,s,p, device=device, **kw))
        blocks.append(nn.Tanh())
        self.layers = nn.Sequential(*blocks)

        for m in self.modules():
            if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d)):
                nn.init.normal_(m.weight,0,.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.normal_(m.weight,1,.02)
                nn.init.zeros_(m.bias)

    def forward(self, x:torch.tensor): 
        return self.layers(cat(x.real, x.imag))

In [2]:
class Generator(nn.Module):
    def __init__(self, device=("cuda:0" if torch.cuda.is_available() else "cpu")):
        super().__init__()
        self.dense = nn.Linear(16*32, 16*16, device=device)
        
        #self.EML_test = EMLConv(3, 3, transpose=True, device=device) #log_channels=1, device=device)
        self.embed_test = ConvEmbedding(1, 3, device=device)
        self.color_test = ColorFocus(3, 1, device=device)
        self.eml_upcycle_test = EMLUpcycle(3, 3*4, 3, 16, 128, device=device)
        self.upcycle_test = CRConvUpcycle(3, 12, 3, 128, 512, device=device)
        """
        self.upcycle1 = EMLConv(3, 3, transpose=True, device=device)
        self.color_2 = ColorFocus(3, 1, device=device)

        self.upcycle2 = EMLConv(3, 3, transpose=True, device=device)
        self.color_3 = ColorFocus(3, 1, device=device)
        """
    def sample_noise(self, batch_size, generator=None):
        x = next(self.parameters())
        return torch.randn(batch_size, 3, 16, 16, device=x.device, dtype=x.real.dtype, generator=generator)

    def forward(self, x:torch.Tensor):
        #display((lambda x: (x.device, x.real.dtype))(next(self.parameters())))
        linear = self.dense(x).reshape(-1, 1, 16, 16)
        embed = self.embed_test(self.sample_noise(x.shape[0]), linear)
        color = self.color_test(embed) #self.EML_test(embed))
        return self.upcycle_test(self.eml_upcycle_test(color))
        #return self.color_3(self.upcycle2(self.color_2(self.upcycle1(color))))
        #return self.EML_test.forward(self.dense(x).reshape(-1, 1, 16, 16))

In [3]:
import clip

device="cuda:0" if torch.cuda.is_available() else "cpu"
model,_=clip.load("ViT-B/32",device=device)
model.eval()
texts=["sdjkcmmskdl","iomkl", 'got', 'to', 'check', 'for', 'batchsize']

with torch.no_grad():
    embeddings = model.encode_text(clip.tokenize(texts).to(device)).float()
embeddings = embeddings.reshape(len(texts),1,16*32)

model.to('cpu')
del(model)

model = Generator()
display((lambda x: (x, x.shape))(model.forward(embeddings).to('cpu')))

/home/nprox/venvs/nprox/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


(tensor([[[[ 1.3120e-02, -6.2515e-02,  4.4095e-02,  ...,  2.8596e-02,
             8.5154e-03,  5.4086e-03],
           [ 3.2125e-02, -2.6871e-02, -1.9303e-02,  ..., -5.1943e-02,
             3.1948e-02,  5.5804e-03],
           [-4.6297e-02, -3.2234e-02, -6.4839e-02,  ...,  6.4914e-02,
            -5.1873e-02, -1.7449e-02],
           ...,
           [ 1.5353e-02,  2.6863e-02,  9.1826e-02,  ...,  3.4110e-02,
            -2.4788e-02,  2.4946e-02],
           [ 4.1027e-02,  1.4296e-01, -1.0768e-01,  ..., -3.0389e-02,
            -6.9112e-02, -2.1026e-03],
           [ 1.3819e-02,  1.0744e-02,  1.3508e-02,  ...,  4.4298e-02,
             6.6652e-03,  1.9031e-02]],
 
          [[ 4.9757e-03, -7.6129e-02,  4.9533e-03,  ..., -5.5401e-02,
            -3.6801e-03, -2.5612e-02],
           [ 4.6712e-03, -7.9608e-03, -1.0805e-01,  ...,  1.8235e-02,
            -2.5545e-02,  1.3630e-02],
           [ 3.9513e-02, -1.2055e-01,  2.7452e-02,  ..., -1.4963e-01,
             1.5891e-02, -6.5423e-02],


In [55]:
batch = 14
#display((lambda x: (x.shape[0], x))(torch.randn(batch, 3, 16, 16)))
display((lambda x: (x.shape, x))(torch.cat((lambda x: (x, x))(torch.randn(batch, 1, 16, 16)), dim=1)))

(torch.Size([14, 2, 16, 16]),
 tensor([[[[-1.0421e+00, -1.7720e-01,  4.0820e-01,  ...,  1.2405e-01,
            -9.0146e-02, -2.4514e-02],
           [ 8.1695e-01, -7.0538e-01,  1.4627e+00,  ...,  2.5500e-01,
            -1.7426e+00, -5.8894e-01],
           [ 3.0652e-01,  1.2059e+00, -1.5159e-02,  ...,  1.1797e+00,
             4.1525e-01,  1.5371e+00],
           ...,
           [ 5.1113e-01,  3.2311e-02, -9.2692e-01,  ...,  6.5543e-01,
            -7.9944e-01,  1.1436e-01],
           [ 3.6585e-01,  7.4254e-01,  4.1843e-01,  ..., -1.4835e+00,
            -1.7237e+00,  5.7614e-02],
           [ 2.3511e+00, -3.1218e-01,  3.7863e-01,  ...,  2.2196e+00,
            -2.2263e-01, -4.2904e-01]],
 
          [[-1.0421e+00, -1.7720e-01,  4.0820e-01,  ...,  1.2405e-01,
            -9.0146e-02, -2.4514e-02],
           [ 8.1695e-01, -7.0538e-01,  1.4627e+00,  ...,  2.5500e-01,
            -1.7426e+00, -5.8894e-01],
           [ 3.0652e-01,  1.2059e+00, -1.5159e-02,  ...,  1.1797e+00,
         

In [ ]:
raise Exception('dflkm')
# %pip install --upgrade --force-reinstall "torch<2.15" torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import pandas as pd
from pathlib import Path
captions = pd.read_json('/home/nprox/AML-DCGAN/yfcc100m_captioned.jsonl', lines=True)
captions['stem'] = captions['filepath'].map(Path).map(lambda x: x.stem)
df = pd.read_json('/home/nprox/datasets/yfcc100m_NL_captions.json')
df['stem'] = df['local_path'].map(Path).map(lambda x: x.stem)
display((lambda x: x[x['captions'].notna()])(pd.merge(df, captions[['stem', 'captions']], on='stem', how='left')))

,local_path,photo_id,caption,processed,stem,captions
196,/mnt/data/yfcc100m/26c/c3b/26cc3bbcdb7d7563133...,4605398871,The wall at the end of the path that goes nowh...,False,26cc3bbcdb7d756313327558975baf3,{'small': 'A long red brick wall stands behind...
2984,/mnt/data/yfcc100m/1ef/73a/1ef73a2f7f39a95eec1...,5376334027,Costa Rica - VBT Highlands and Coast. VBT Walk...,False,1ef73a2f7f39a95eec1d7efce52f3dbd,{'small': 'A scenic coastal view features a ca...
3195,/mnt/data/yfcc100m/26a/b34/26ab3429af0b57cac10...,5558545636,Fondren After 5. The art on display in Brown's...,False,26ab3429af0b57cac10ad1c15f45868,{'small': 'A blonde woman in a black fur coat ...
3290,/mnt/data/yfcc100m/dcb/4e9/dcb4e9e4ea0f3b4adae...,6623674565,Santa's coming!...big snow of December 2010...,False,dcb4e9e4ea0f3b4adaefec5c26ac67d,{'wild': 'A LEGO Santa Claus figure stands in ...
3565,/mnt/data/yfcc100m/a9a/7e6/a9a7e634c197fac12c3...,3183982448,Grillparty. The grill.,False,a9a7e634c197fac12c3cf1b253dd32b9,{'small': 'A man in a red shirt cooks food on ...
...,...,...,...,...,...,...
8694775,/mnt/data/yfcc100m/c16/9d2/c169d2cc2577c6451d1...,6465357021,"President Van Rompuy talks to Nikola Gruevski,...",False,c169d2cc2577c6451d1a53c1d5cc1280,{'small': 'Two men in suits sit in armchairs c...
8694991,/mnt/data/yfcc100m/f55/7c8/f557c875f633b6caebc...,54227323,New work uniform. This is my new work uniform:...,False,f557c875f633b6caebc06641729e35ac,"{'small': 'A smiling man wears a black ""Lomba ..."
8695608,/mnt/data/yfcc100m/5af/f33/5aff334f8cb5fd9977c...,6345432874,Greene King Brewery Tour 28-07-2011. Tour the ...,False,5aff334f8cb5fd9977cfa8d5ea27fbdd,{'wild': 'A person in historical costume kneel...
8695887,/mnt/data/yfcc100m/b95/a41/b95a41a151261edf9c7...,291875673,First snow on the Galen mountain. First snow o...,False,b95a41a151261edf9c7b1f34e4672035,{'small': 'A snow-capped mountain rises majest...


In [ ]:
import pandas as pd
from pathlib import Path
data = pd.read_csv('//home//nprox//datasets//yfcc100m_subset_data.tsv', names=['line_number', 'photo_id', 'photo_hash'], header=None, sep='\t')
folders = data['photo_hash'].map(lambda x: x[:3]).unique()

drives_dir = Path('/mnt') #Path(args.drives_dir)
path_map = {fldr.name: Path(fldr) 
            for drive in drives_dir.iterdir() 
            for fldr in (drive/'yfcc100m').iterdir() 
            if fldr.name in folders}
# all photos .jpg
dwnld_photos = data['photo_hash'].map(lambda x: path_map.get(x[:3], Path())/Path(x[3:6])/Path(x+'.jpg') if path_map.get(x[:3], False) != False else pd.NA).dropna()
dwnld_photos = pd.DataFrame(dwnld_photos.to_numpy(), columns=['local_path'])
#dwnld_photos = dwnld_photos[dwnld_photos.map(Path.exists)] #.map(lambda x: x if x.exists() else pd.NA).dropna()
dwnld_photos['photo_hash'] = dwnld_photos['local_path'].map(lambda x: x.stem)
dwnld_photos = pd.merge(dwnld_photos, data[['photo_hash', 'photo_id']], on='photo_hash', how='left')
"""
dwnld = [fldr.name for drive in drives_dir.iterdir() 
        for fldr in (drive/'yfcc100m').iterdir() if fldr.name in folders]
to_dwnld = [fldr for fldr in folders if fldr not in dwnld]"""


"\ndwnld = [fldr.name for drive in drives_dir.iterdir() \n        for fldr in (drive/'yfcc100m').iterdir() if fldr.name in folders]\nto_dwnld = [fldr for fldr in folders if fldr not in dwnld]"

In [ ]:
dwnld_photos['photo_hash'].str[:3].unique()

<StringArray>
['b61', '4be', 'faa', 'ee4', 'bb6', 'a42', 'bd6', '4aa', 'de8', '47b',
 ...
 '06f', '009', '0ae', '08d', '00a', '00e', '005', '003', '00c', '001']
Length: 2359, dtype: str

In [ ]:
import sqlite3

con=sqlite3.connect('file:/home/nprox/datasets/yfcc100m_dataset.sql?mode=ro',uri=True)
con.execute('''
    CREATE TEMP TABLE IF NOT EXISTS photoid (
        photoid INTEGER PRIMARY KEY
    )
''')
con.execute('DELETE FROM temp.photoid')

con.executemany(
    'INSERT OR IGNORE INTO temp.photoid(photoid) VALUES (?)',
    ((int(x),) for x in dwnld_photos['photo_id'])
)
con.commit()

df = pd.read_sql_query('''
    SELECT y.photoid, y.title, y.description
    FROM temp.photoid AS w
    JOIN main.yfcc100m_dataset AS y ON y.photoid = w.photoid
''', con)

In [ ]:
from urllib.parse import unquote_plus
from bs4 import BeautifulSoup

dwnld_photos = (lambda x: x[[y for y in x.columns if y != 'photoid']])(pd.merge(dwnld_photos, df, left_on='photo_id', right_on='photoid'))
dwnld_photos['title'] = dwnld_photos['title'].map(unquote_plus)
dwnld_photos['description'] = dwnld_photos['description'].map(unquote_plus)
dwnld_photos.loc[dwnld_photos['description'].str.contains('<|>', regex=True), 'description'] = \
    dwnld_photos[dwnld_photos['description'].str.contains('<|>', regex=True)]['description'].map(lambda x: BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')
dwnld_photos.loc[~dwnld_photos['title'].str.contains(r'\.$'), 'title'] = dwnld_photos[~dwnld_photos['title'].str.contains(r'\.$')]['title'] + '.'
dwnld_photos['title'] = dwnld_photos['title'].str.replace(r'\.+\W*', '. ')
dwnld_photos['caption'] = dwnld_photos['title'] + dwnld_photos['description']
dwnld_photos

,local_path,photo_hash,photo_id,title,description
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken,recipe here: allrecipes.com/Recipe/Denises-Pea...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer,The Arches National Park is a wonderland of ge...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy,Review: Chong Yen (Steam Fish) Food Stall
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1,ME White High Collar Thobe
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3),Stars Vs. Canucks - 2/5/08 - the Stars and Can...
...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011,A tour to the Geysers geothermal field in Cali...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light,"Part of the set ""Ravine Views"" - there is a ra..."
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos,30th Anniversary at Bowery Electric. Featuring...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather,Photos taken by my dad at my graduation (from ...


In [ ]:
import os
str(Path(os.getcwd()))

'/home/nprox/AML-DCGAN'

In [ ]:
dwnld_photos

,local_path,photo_hash,photo_id,title,description,caption
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken.,recipe here: allrecipes.com/Recipe/Denises-Pea...,denise's peanut chicken. recipe here: allrecip...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer.,The Arches National Park is a wonderland of ge...,spot the intrepid explorer. The Arches Nationa...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy.,Review: Chong Yen (Steam Fish) Food Stall,Yau Choy. Review: Chong Yen (Steam Fish) Food ...
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1.,ME White High Collar Thobe,ME White High Collar Thobe 1. ME White High Co...
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3).,Stars Vs. Canucks - 2/5/08 - the Stars and Can...,The Fight (step 3). Stars Vs. Canucks - 2/5/08...
...,...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011.,A tour to the Geysers geothermal field in Cali...,The Geysers - Tour in February 2011. A tour to...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light.,"Part of the set ""Ravine Views"" - there is a ra...",Ravine April 9 - Afternoon Light. Part of the ...
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos.,30th Anniversary at Bowery Electric. Featuring...,The Waldos. 30th Anniversary at Bowery Electri...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather.,Photos taken by my dad at my graduation (from ...,My Grandfather. Photos taken by my dad at my g...


In [ ]:
dwnld_photos['caption'] = dwnld_photos['title'] + " " + dwnld_photos['description']

In [ ]:
from bs4 import BeautifulSoup

dwnld_photos.loc[dwnld_photos['description'].str.contains('<|>', regex=True), 'description'] = dwnld_photos[dwnld_photos['description'].str.contains('<|>', regex=True)]['description'].map(lambda x: BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')
dwnld_photos

,local_path,photo_hash,photo_id,title,description
0,/mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...,b61f252678da02eaff453bce575cb6,2297552664,denise's peanut chicken,recipe here: allrecipes.com/Recipe/Denises-Pea...
1,/mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...,4be1ac2e9425d56e9140baa8457c399,2932067831,spot the intrepid explorer,The Arches National Park is a wonderland of ge...
2,/mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...,faa524f25942ba9362db7495a61651c,5177649120,Yau Choy,Review: Chong Yen (Steam Fish) Food Stall
3,/mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...,ee42a0b73fc6753b877131e84b39fb,9743459054,ME White High Collar Thobe 1,ME White High Collar Thobe
4,/mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...,bb68d7fee09d1ef0837eb3a4d2c8ab8c,2245962407,The Fight (step 3),Stars Vs. Canucks - 2/5/08 - the Stars and Can...
...,...,...,...,...,...
8561554,/mnt/data/yfcc100m/364/31c/36431c5c8342a26a29d...,36431c5c8342a26a29de28a5ec98926e,5786441045,The Geysers - Tour in February 2011,A tour to the Geysers geothermal field in Cali...
8561555,/mnt/data/yfcc100m/3d8/739/3d8739fa25813e32659...,3d8739fa25813e3265958fd9437f58,4506571174,Ravine April 9 - Afternoon Light,"Part of the set ""Ravine Views"" - there is a ra..."
8561556,/mnt/data2/yfcc100m/19d/5f8/19d5f84ebf86d3d31a...,19d5f84ebf86d3d31ae3e2c094a1733d,7179040256,The Waldos,30th Anniversary at Bowery Electric. Featuring...
8561557,/mnt/data2/yfcc100m/739/2df/7392df6fd6a3a37ac7...,7392df6fd6a3a37ac7b9bd517a591e,4899784138,My Grandfather,Photos taken by my dad at my graduation (from ...


In [ ]:
dwnld_photos[~dwnld_photos['description'].str.contains('<')].shape[0] - ((lambda x: x[~x.str.contains('<')])(dwnld_photos['description']).map(unescape)).shape[0]

0

In [ ]:
dwnld_photos['description'].map(lambda x: x if '<' not in x else BeautifulSoup(x, 'lxml').get_text(' ', strip=True), na_action='ignore')

KeyboardInterrupt: 

In [ ]:
from html import unescape
from bs4 import BeautifulSoup
(lambda x: x[~x[0].str.contains('<a href="(?:.*rel="nofollow")?|^<.?.?>|^</?del>|^</?strong>')][0].map(lambda x: BeautifulSoup(unescape(x), 'lxml').get_text(' ', strip=True)))(dwnld_photos['description'].str.extractall(r'(<.*>)').dropna())

         match
1035     0                            http://ClintJCL.wordpress.com
1491     0                                         Sissel - One Day
2496     0            http://www.day-trips.ca/content/niagara-falls
2511     0                            http://ClintJCL.wordpress.com
2937     0                                       Read the Blog Post
                                        ...                        
8560931  4                                                 Part III
         5        Bradley Bodiford, a Pi Kappa Phi at Mercer Uni...
         6        Josh Rhodes explains Pi Kappa Phi's Build Amer...
8561175  0                                                 On black
8561285  0                                       PopCultureGeek.com
Name: 0, Length: 16045, dtype: str

In [ ]:
(lambda x: (lambda y: y[~y[0].str.contains('<..?>', regex=True)])(x[~x[0].str.contains('<a href="')]))(dwnld_photos['description'].str.extractall(r'(<.*>)').dropna())

0
        match                                                   
667     0      <strong>More photos, videos e the setlist: </s...
1685    0      <del>The museum was closing when I took this p...
2514    1           <strong>Find The Cosmopolitan on...</strong>
3994    0                                      <strong></strong>
4521    2                                               <strong>
...                                                          ...
8555516 7                              <strong>Library:</strong>
8556310 0                           <strong>the shot...</strong>
        1                          <strong>and then... </strong>
        2                          <strong>photoshop...</strong>
8557507 0       <strong>New Travel Guide and Photo-map:</strong>

[16593 rows x 1 columns]

In [ ]:

"""
con.execute(
 'SELECT photoid, title,description FROM yfcc100m_dataset WHERE photoid=?',
 (int(dwnld_photos['photo_id'].iloc[234]),)
).fetchall()
"""
df = pd.DataFrame(con.execute(f'SELECT photoid, title,description FROM yfcc100m_dataset').fetchall(), columns=['photo_id', 'title', 'description'])

In [ ]:
dwnld_photos.iloc[:10000].map(lambda x: next(x.parent.glob(f'{x.stem}*'), "None")).dropna()

0        /mnt/data/yfcc100m/b61/f25/b61f252678da02eaff4...
2        /mnt/data/yfcc100m/4be/1ac/4be1ac2e9425d56e914...
3        /mnt/data/yfcc100m/faa/524/faa524f25942ba9362d...
4        /mnt/data2/yfcc100m/ee4/2a0/ee42a0b73fc6753b87...
5        /mnt/data/yfcc100m/bb6/8d7/bb68d7fee09d1ef0837...
                               ...                        
17570    /mnt/data/yfcc100m/183/23c/18323cae5642cc667e7...
17571    /mnt/data/yfcc100m/3d8/eab/3d8eabf1c832d892fff...
17572    /mnt/data/yfcc100m/797/9f1/7979f1329561ae78197...
17573    /mnt/data2/yfcc100m/86a/538/86a538c78e7ad36814...
17574    /mnt/data2/yfcc100m/77f/be0/77fbe06f28151fdbf3...
Name: photo_hash, Length: 9997, dtype: object

In [ ]:
path_map = {fldr.name: Path(fldr) 
            for drive in drives_dir.iterdir() 
            for fldr in (drive/'yfcc100m').iterdir() 
            if fldr.name in folders}
dwnld_photos = data['photo_hash'].map(lambda x: next((path_map.get(x[:3], Path())/Path(x[3:6])).glob(f'{x}*'), None) if path_map.get(x[:3], False) != False else pd.NA).dropna()

KeyboardInterrupt: 

In [ ]:
import os
dwnld_photos.map(lambda x: os.path.exists(x)).value_counts()

KeyboardInterrupt: 

In [ ]:
display(len({1:2, 3:4}))
raise Exception('sdjnclksdm')

2

Exception: sdjnclksdm

In [ ]:
from pathlib import Path
import numpy as np
import subprocess
import argparse
import shutil
import ollama
import json
import sys
import re
import os

# %pip install git+https://github.com/openai/CLIP.git
#import clip
#from clip.simple_tokenizer import SimpleTokenizer
"""
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--drives-dir", type=Path, default="/mnt")
parser.add_argument("--out-dir", type=Path, default=os.getcwd())
parser.add_argument("--parallel", type=int, default=18)
parser.add_argument("--model", type=str, default='qwen3.8-27b:q4_k_m')
args = parser.parse_args()"""

yfcc100m = []
drives_dir = Path('/mnt') #Path(args.drives_dir)
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1)
    if not os.path.exists(drive_dir / 'yfcc100m'):
        continue
    drive_dir = drive_dir / 'yfcc100m'
    yfcc100m.extend([Path(sub_fldr) for fldr in drive_dir.iterdir() for sub_fldr in fldr.iterdir() \
                     if (fldr/'.complete').is_file() and not (sub_fldr/'.parsed').is_file() and \
                        not re.match(r'.*/.complete$', str(sub_fldr))])
indices = np.random.permutation(np.arange(yfcc100m.__len__()))

PROMPT = ["Describe this image in one concise English sentence, at most 25 words. Mention the main subject, action, and setting when visible. Return only the caption.", \
          "Return only a caption",]
subprocess.run(['llama-server', f'-hf bartowski/{args.model}}', '--n-gpu-layers 99', f'--parallel {args.parallel}', 
            '--ctx-size 8192', '--cont-batching', '--jinja', '--reasoning off', '--alias yfcc-captioner', 
            '--host 127.0.0.1', '--port 8080'], capture_output=False, text=True, check=True,)


# batch then add to whole
fldr = Path(os.getcwd()) # Path(args.out_dir)
for idx in indices:
    batch = ''
    for img in Path(yfcc100m[idx]).iterdir():
        # construct json in mem? time diff?
        if not img.is_file() or img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        response = ollama.chat(
            model=args.model,
            messages=[{
                "role": "user",
                "content": PROMPT,
                "images": [img.read_bytes()],
            }],
            think=False,
            stream=False,
            keep_alive="30m",
            options={
                "num_ctx": 4096,
                "num_predict": 77,
                "temperature": 0.2,
            },
        )

        caption = response.message.content.strip()
        if not caption or response.done_reason == "length":
            raise RuntimeError(f"Empty or truncated caption for {img}; request a shorter caption.")
        batch += caption
    (yfcc100m['idx']/'.parsed').touch()
    (fldr/'yfcc100m_captioned.jsonl').touch()
    with (fldr/'yfcc100m_captioned.jsonl').open("a", encoding="utf-8") as file:
        file.write(json.dumps(batch, ensure_ascii=False, default=str) + "\n")


/mnt/data/yfcc100m/8f3/7e0/8f37e055bded36339ef06ca164a44bbd.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e017ab4c5f3f3e27ec705d263675.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0a85b959c51ae8d24cfb246b3a.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0bece8ce226f588fe680268cfd.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e0254b2b31d7d4f4adae27dbde4.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e037ee32867bac39ab706e5a66e7.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e034ffa3ea63139c34cd7a4bab52.jpg
/mnt/data/yfcc100m/8f3/7e0/8f37e01af858a17851ebbc645dfdd3.jpg


In [ ]:


fldr = Path.cwd()  # Or Path(args.out_dir)
output = fldr / "yfcc100m_captioned.jsonl"

for idx in indices:
    batch = fldr / "batch.jsonl"
    # Reset the temporary batch and keep it open for this subfolder.
    with batch.open("w", encoding="utf-8") as file:
        for img in source.iterdir():
            

            value = {"image": str(img), "caption": caption}
            file.write(json.dumps(value, ensure_ascii=False) + "\n")

    # Each batch line is already JSON; append its text directly.
    with batch.open("r", encoding="utf-8") as src, \
         output.open("a", encoding="utf-8") as dst:
        shutil.copyfileobj(src, dst)

    (source / ".parsed").touch()

In [ ]:
MODEL = "qwen3.8:27b-q4_k_m"
PROMPT = (
    "Describe this image in one concise English sentence, at most 25 words. "
    "Mention the main subject, action, and setting when visible. "
    "Return only the caption."
)

response = ollama.chat(
                model=MODEL,
                messages=[{
                    "role": "user",
                    "content": 'return only a caption', #PROMPT,
                    "images": [Path('/mnt/data/yfcc100m/8f3/7e0/8f37e01af858a17851ebbc645dfdd3.jpg').read_bytes()],
                }],
                think=False,
                stream=False,
                keep_alive="30m",
                options={
                    "num_ctx": 4096,
                    "num_predict": 77,
                    "temperature": 0.2,
                },
            )

caption = response.message.content.strip()
caption

'A smiling woman with short dark hair, wearing a white long-sleeved athletic top and black shorts, stands indoors with her arms crossed, turning to look over her shoulder at the camera. She has a race bib or event tag pinned to the back of her shirt. The room features light green walls with wood paneling, a wooden chair, and a gray upholstered armchair.'

In [ ]:
yfcc100m.__len__()

237486

In [ ]:
for subfldr in Path('/mnt/data/yfcc100m/365').iterdir():
    if :
        print(subfldr)
yfcc100m.__len__()
os.getcwd()

/mnt/data/yfcc100m/365/.complete


'/home/nprox/AML-DCGAN'

In [ ]:
%tb
raise Exception('sdjkfnvlkmfd')

SystemExit: 2

Exception: sdjkfnvlkmfd

In [ ]:
from pathlib import Path
import numpy as np
import subprocess
import shutil
import re
import os

yfcc100m = []
drives_dir = Path('/mnt/')
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1)
    if not os.path.exists(drive_dir / 'yfcc100m'):
        os.mkdir(drive_dir / 'yfcc100m')
    drive_dir = drive_dir / 'yfcc100m'
    yfcc100m.extend([re.match(r'.*/([^/]+)$', str(fldr)).group(1) for fldr in drive_dir.iterdir()])

folders = subprocess.run(["aws", "s3", "ls", "s3://multimedia-commons/data/images/", "--no-sign-request", "--region", "us-west-2",], \
                        capture_output=True, text=True, check=True,).stdout.splitlines()
folders = [fdr for line in folders if (fdr := re.match(r".*PRE (.+)/", line).group(1)) not in yfcc100m]
indices = np.random.permutation(np.arange(folders.__len__()))

cntr = 0
for drive in drives_dir.iterdir():
    drive_dir = drives_dir / re.match(r'.*/([^/]+)$', str(drive)).group(1) / 'yfcc100m'
    for idx in indices[cntr:].copy():
        if shutil.disk_usage(drive_dir)[-1] < 5*1024**3:
            break
        print(folders[idx])
        dwnl = subprocess.run(["aws", 's3', 'sync', f's3://multimedia-commons/data/images/{folders[idx]}/',
                               (drive_dir/folders[idx]), '--no-sign-request', '--region', 'us-west-2',],)
        (drive_dir/folders[idx]).touch()
        cntr += 1

array([2311, 1657, 2197, ..., 2082, 1246,  798], shape=(4094,))

In [ ]:
dwnl = subprocess.run(["aws", 's3', 'sync', f's3://multimedia-commons/data/images/{folders[idx]}',
                        (drive_dir/folders[indices[100]]), '--no-sign-request', '--region', 'us-west-2',], \
                        text=True, check=True)

In [ ]:
np.array(folders)[idx]

array(['5d3', '194', '009', ..., 'c35', 'a78', '2e1'],
      shape=(4081,), dtype='<U3')

array(['000', '001', '002', ..., 'fee', 'fef', 'ff0'],
      shape=(4081,), dtype='<U3')